- [Steps]()
    - [Sine]()
- [Sine]()
    - [Steps]()

In [1]:
import os
dir = os.path.abspath('')  # directory of notebook
import sys
sys.path.append(os.path.join(dir, '..', '..'))  # package directory
from src.FoKL import FoKLRoutines
from src.FoKL.fokl_to_pyomo import fokl_to_pyomo
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import pyomo.dae as dae
from scipy.interpolate import interp1d

In [17]:
filename = [os.path.join(dir, "data", "step_tests.csv"), 
            os.path.join(dir, "data", "tclab_sine_test.csv")]

steps = pd.read_csv(filename[0])
sine = pd.read_csv(filename[1])

# Known parameters:
alpha = 0.00016 # watts / (units P1 * percent U1)
P1 = 200 # P1 units

In [20]:
def parse(data):
    """'data' is a pd.DataFrame. Returned is 'tvec', 'Ts1', 'u1'."""
    return data["Time"], data["T1"] - data["T1"][0], data["Q1"]


def params(tvec, Ts1, u1, param0=None):
    """Optimize parameters for mechanistic model. Returned is 'Ua', 'Ub', 'CpH', 'CpS'."""
    if param0 is None:  # initial point
        param0 = [0.05,  # Ua, watts/deg C
                 0.05,  # Ub, watts/deg C
                 5,     # CpH, joules/deg C
                 1]     # CpS, joules/deg C

    # Solve model:

    u1_interp = interp1d(tvec, u1, kind='previous')

    def tclab_model(param):
        Ua, Ub, CpH, CpS = param  # adjustable parameters

        def deriv(t, y):  # two-state model
            T1H, T1S = y
            dT1H = (-Ua * T1H + Ub * (T1S - T1H) + alpha * P1 * u1_interp(t)) / CpH
            dT1S = Ub * (T1H - T1S) / CpS
            return [dT1H, dT1S]
        
        soln = solve_ivp(deriv, [tvec[0], tvec[-1]], [0, 0], t_eval=tvec)  # model solution
        
        pred = pd.DataFrame()  # dataframe with predictions
        pred["Time"] = tvec
        pred["TH1"] = soln.y[0]
        pred["TS1"] = soln.y[1]
        pred["Q1"] = u1_interp(tvec)

        return pred

    # Regress model
    def residuals(param):
        pred = tclab_model(param)
        return pred["TS1"] - Ts1
    
    # Set bounds (non-negative):
    bnds = ([0.0, 0.0, 0.0, 0.0], [np.inf, np.inf, np.inf, np.inf])

    # Perform least squares nonlinear regression:
    nl_results = least_squares(residuals, param0, bounds=bnds, method='trf', verbose=2, loss="arctan")

    param = nl_results.x
    return param, tclab_model(param)["TS1"], tclab_model(param)["TH1"]  # == [Ua, Ub, CpH, CpS], Ts1_prediction, Th1_prediction


def smooth(x, window):
    """Apply centered average of size window."""
    x_smooth = np.zeros_like(x)
    w2 = int(np.floor(window / 2))
    w2p1 = w2 + 1

    # bleed in:
    for i in range(w2):
        x_smooth[i] = np.mean(x[:(i + w2p1)])

    # center:
    for i in range(w2, x_smooth.size - w2):
        x_smooth[i] = np.mean(x[(i - w2):(i + w2p1)])

    # bleed out:
    for i in range(-w2, 0):
        x_smooth[i] = np.mean(x[(i - w2)::])

    return x_smooth


def gradient_h4(x, h):
    """h is step size. Order of error is h^4."""
    dx = np.zeros_like(x)

    # bleed in:
    h2 = 2 * h
    dx[0] = (x[1] - x[0]) / h
    dx[1] = (x[2] - x[0]) / h2

    # center difference:
    h12 = 12 * h
    for i in range(2, x.shape[0] - 2):
        dx[i] = (x[i - 2] - 8 * x[i - 1] + 8 * x[i + 1] - x[i + 2]) / h12
    
    # bleed out:
    dx[-2] = (x[-1] - x[-3]) / h2
    dx[-1] = (x[-1] - x[-2]) / h

    return dx

In [19]:
# Cycle through 'steps' and 'sine' datasets:
for data in [steps, sine]:
    
    # Parse datasets:
    tvec, Ts1, u1 = parse(data)
    n = len(tvec)
    dt = (tvec[-1] - tvec[0]) / (n - 1)
    
    # Optimize two-state model parameters:
    param, Ts1_p, Th1_p = params(tvec, Ts1, u1)

    # Differentiate smoothed data:
    dTs1 = gradient_h4(smooth(Ts1, 9), dt)
    dTs1_p = gradient_h4(Ts1_p, dt)  # Ts1_p already smooth
    
    # Derivative of residual:
    dR = dTs1 - dTs1_p

    # Train GP on un-corrected (Ts1, Th1) and measured u1:

    

   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         5.5405e+02                                    6.97e+04    
       1              6         4.3414e+02      1.20e+02       9.07e-03       2.30e+04    
       2              7         3.6673e+02      6.74e+01       3.80e-02       6.91e+03    
       3              8         3.2434e+02      4.24e+01       3.99e-02       1.07e+04    
       4              9         2.8035e+02      4.40e+01       8.08e-02       6.86e+03    
       5             10         2.5676e+02      2.36e+01       1.59e-01       2.40e+03    
       6             11         1.4862e+02      1.08e+02       1.59e-01       2.52e+03    
       7             12         1.3887e+02      9.75e+00       2.17e-01       1.48e+03    
       8             18         1.3714e+02      1.73e+00       8.14e-05       1.08e+03    
       9             20         1.3651e+02      6.38e-01       3.65e-05       2.38e+03    